# VisionBridge — trained model check (Colab)

**Inference/checking notebook only.** It does not train the model or modify VisionBridge source code.

It downloads one real sentence-level ISL-CSLTR video, extracts the same 132-dimensional pose + 1404-dimensional face features used by VisionBridge, and runs the already-trained checkpoint through the real CTC decoder.

The training notebook is intentionally untouched.


In [ ]:
# Step 1 — Bootstrap repo and imports
import os, sys, subprocess, shutil
from pathlib import Path

REPO_ROOT = Path("/content/VisionBridge")
if not (REPO_ROOT / "README.md").exists():
    subprocess.run(["git", "clone", "https://github.com/BharathWaj-K-R/VisionBridge.git", str(REPO_ROOT)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_ROOT), "pull", "--ff-only"], check=True)

BACKEND_ROOT = REPO_ROOT / "backend"
if str(BACKEND_ROOT) not in sys.path:
    sys.path.insert(0, str(BACKEND_ROOT))
os.chdir(REPO_ROOT)

import app

print("Repo:", REPO_ROOT)
print("Python:", sys.version.split()[0])
print("APP IMPORT: PASS")


## Step 2 — Prepare isolated Python 3.12 + MediaPipe 0.10.21

Colab can run Python 3.13, while the legacy MediaPipe Holistic API used by this repository is not available from newer MediaPipe packages. We therefore keep the main Colab kernel untouched and run **only keypoint extraction** in a separate Python 3.12 environment.


In [ ]:
# Step 2 — Create/verify the isolated extractor environment.
import importlib.util
import subprocess
import sys
import shutil
from pathlib import Path

MP_ENV = Path("/content/visionbridge_mp312")
MP_PYTHON = MP_ENV / "bin" / "python"

def run(cmd):
    print("$", " ".join(map(str, cmd)))
    return subprocess.run(cmd, check=True, text=True)

uv_bin = shutil.which("uv")
if uv_bin is None:
    run([sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir", "uv"])
    uv_bin = shutil.which("uv")

if uv_bin is None:
    raise RuntimeError("uv was installed but its executable is not on PATH. Restart Colab and rerun this cell.")

probe_py = subprocess.run([uv_bin, "python", "find", "3.12"], text=True, capture_output=True)
if probe_py.returncode != 0:
    run([uv_bin, "python", "install", "3.12"])

if not MP_PYTHON.exists():
    run([uv_bin, "venv", "--python", "3.12", str(MP_ENV)])

probe = subprocess.run(
    [str(MP_PYTHON), "-c", "import mediapipe; from mediapipe.python.solutions import holistic; print(mediapipe.__version__)"],
    text=True, capture_output=True,
)

if probe.returncode != 0 or probe.stdout.strip() != "0.10.21":
    run([uv_bin, "pip", "install", "--python", str(MP_PYTHON), "mediapipe==0.10.21", "numpy==1.26.4", "opencv-python-headless", "pandas"])

probe = subprocess.run(
    [str(MP_PYTHON), "-c", "import sys, mediapipe; from mediapipe.python.solutions import holistic; print('Python:', sys.version.split()[0]); print('MediaPipe:', mediapipe.__version__); print('Legacy Holistic import: PASS')"],
    text=True, capture_output=True,
)

print(probe.stdout)
if probe.returncode != 0:
    print(probe.stderr)
    raise RuntimeError("Isolated Python 3.12 MediaPipe environment failed validation.")


In [ ]:
# Step 3 — Load the trained checkpoint.
import torch

WEIGHTS = REPO_ROOT / "backend/app/models/weights/base_model.pt"
VOCAB = REPO_ROOT / "backend/app/models/weights/base_model.vocab.json"

if not WEIGHTS.exists() or not VOCAB.exists():
    from google.colab import files
    print("Upload BOTH base_model.pt and base_model.vocab.json")
    uploaded = files.upload()
    for name in ("base_model.pt", "base_model.vocab.json"):
        if name not in uploaded:
            raise FileNotFoundError(f"Missing required artifact: {name}")
        WEIGHTS.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy(name, WEIGHTS.parent / name)

assert WEIGHTS.exists(), f"Missing weights: {WEIGHTS}"
assert VOCAB.exists(), f"Missing vocabulary: {VOCAB}"

from app.training.isltranslate import SimpleCharTokenizer
from app.models.base_model import load_frozen_base_model, POSE_INPUT_DIM, FACE_INPUT_DIM, MAX_SEQUENCE_LENGTH

tokenizer = SimpleCharTokenizer.load(VOCAB)
state = torch.load(WEIGHTS, map_location="cpu")
assert isinstance(state, dict), "Checkpoint is not a state-dict dictionary."
assert "output_head.weight" in state, "Missing output_head.weight."
checkpoint_vocab = int(state["output_head.weight"].shape[0])
assert checkpoint_vocab == tokenizer.vocab_size, (
    f"Vocabulary mismatch: checkpoint={checkpoint_vocab}, tokenizer={tokenizer.vocab_size}"
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = load_frozen_base_model(str(WEIGHTS), vocab_size=tokenizer.vocab_size).to(device).eval()
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
assert trainable == 0, "Base model is not frozen."

print("Device:", device)
print("Vocabulary:", tokenizer.vocab_size)
print("Trainable parameters:", trainable)
print("CHECKPOINT VALIDATION: PASS")


## Step 4 — Forward-pass smoke test

Synthetic zeros are used only to verify the model tensor contract. They are not an accuracy test.


In [ ]:
frames = 16
pose = torch.zeros(1, frames, POSE_INPUT_DIM, device=device)
face = torch.zeros(1, frames, FACE_INPUT_DIM, device=device)
with torch.inference_mode():
    logits = model(pose, face)

assert tuple(logits.shape) == (1, frames, tokenizer.vocab_size)
print("Pose:", tuple(pose.shape))
print("Face:", tuple(face.shape))
print("Logits:", tuple(logits.shape))
print("FORWARD TEST: PASS")


## Step 5 — Download the real ISL-CSLTR dataset

This downloads the real sentence-level video dataset only for an inference check. It does not train anything.


In [ ]:
import glob
import importlib.util
import os
import subprocess
import sys

if importlib.util.find_spec("kagglehub") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir", "kagglehub"], check=True)

import kagglehub
dataset_path = kagglehub.dataset_download("drblack00/isl-csltr-indian-sign-language-dataset")
print("Dataset path:", dataset_path)

roots = [
    p for p in glob.glob(os.path.join(dataset_path, "**", "*Sentence_Level*"), recursive=True)
    if os.path.isdir(p) and "Video" in os.path.basename(p)
]

if len(roots) != 1:
    print("Candidate sentence-level directories:")
    for root in roots:
        print(" ", root)
    raise RuntimeError(f"Expected exactly one sentence-level video directory, found {len(roots)}.")

VIDEO_ROOT = roots[0]
video_files = []
for ext in ("*.mp4", "*.MP4", "*.avi", "*.AVI", "*.mov", "*.MOV"):
    video_files.extend(glob.glob(os.path.join(VIDEO_ROOT, "**", ext), recursive=True))
video_files = sorted(video_files)
assert video_files, f"No sentence-level videos found under {VIDEO_ROOT}"
print("Video root:", VIDEO_ROOT)
print("Videos found:", len(video_files))


## Step 6 — Select one real sentence-level video

In [ ]:
TEST_VIDEO = video_files[0]
expected_text = Path(TEST_VIDEO).parent.name.replace("_", " " ).strip()
print("Selected video:", TEST_VIDEO)
print("Expected label:", expected_text)


## Step 7 — Extract real keypoints in isolated Python 3.12

This uses the repository's existing `backend/scripts/extract_keypoints.py`. A helper subprocess runs only the extractor in the isolated environment, then the main Colab kernel loads the generated arrays.


In [ ]:
import subprocess
import textwrap

CHECK_DIR = REPO_ROOT / "data/model_check"
CHECK_DIR.mkdir(parents=True, exist_ok=True)
pose_path = CHECK_DIR / "pose.npy"
face_path = CHECK_DIR / "face.npy"
helper = CHECK_DIR / "_extract_one.py"

helper.write_text(textwrap.dedent("""
import sys
from pathlib import Path
repo = Path(sys.argv[1])
video = sys.argv[2]
pose_out = Path(sys.argv[3])
face_out = Path(sys.argv[4])
sys.path.insert(0, str(repo / "backend"))
import numpy as np
import mediapipe
from mediapipe.python.solutions import holistic
from scripts.extract_keypoints import extract_clip_keypoints
with holistic.Holistic(static_image_mode=False, model_complexity=1) as solution:
    pose, face = extract_clip_keypoints(video, solution)
assert pose.ndim == 2 and pose.shape[1] == 132, pose.shape
assert face.ndim == 2 and face.shape[1] == 1404, face.shape
assert pose.shape[0] == face.shape[0] and pose.shape[0] > 0
np.save(pose_out, pose)
np.save(face_out, face)
print("Python:", sys.version.split()[0])
print("MediaPipe:", mediapipe.__version__)
print("POSE_SHAPE:", pose.shape)
print("FACE_SHAPE:", face.shape)
print("REAL_KEYPOINT_EXTRACTION: PASS")
""""), encoding="utf-8")

result = subprocess.run(
    [str(MP_PYTHON), str(helper), str(REPO_ROOT), TEST_VIDEO, str(pose_path), str(face_path)],
    text=True, capture_output=True,
)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError("Real-video keypoint extraction failed.")

assert pose_path.exists() and face_path.exists()


## Step 8 — Run the trained model on the real sequence


In [ ]:
import numpy as np
from app.services.inference_service import decode_logits
from app.training.isltranslate import _downsample_to_max_length

pose_np = np.load(pose_path)
face_np = np.load(face_path)
assert pose_np.ndim == 2 and pose_np.shape[1] == 132
assert face_np.ndim == 2 and face_np.shape[1] == 1404
assert pose_np.shape[0] == face_np.shape[0] and pose_np.shape[0] > 0

pose_t, face_t = _downsample_to_max_length(
    torch.from_numpy(pose_np).float(),
    torch.from_numpy(face_np).float(),
    Path(TEST_VIDEO).stem,
)
assert pose_t.shape[0] <= MAX_SEQUENCE_LENGTH

with torch.inference_mode():
    logits = model(pose_t.unsqueeze(0).to(device), face_t.unsqueeze(0).to(device))
prediction, confidence = decode_logits(logits)

print("\n" + "=" * 60)
print("REAL VISIONBRIDGE PREDICTION")
print("=" * 60)
print("VIDEO:", Path(TEST_VIDEO).name)
print("GROUND TRUTH:", expected_text)
print("PREDICTED:", prediction)
print("CONFIDENCE:", round(float(confidence), 4))
print("FRAMES USED:", pose_t.shape[0])
print("LOGITS:", tuple(logits.shape))
print("=" * 60)


## Step 9 — Calculate CER

In [ ]:
def levenshtein(a, b):
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cur = [i]
        for j, cb in enumerate(b, 1):
            cur.append(min(cur[-1] + 1, prev[j] + 1, prev[j - 1] + (0 if ca == cb else 1)))
        prev = cur
    return prev[-1]

truth = expected_text.lower().strip()
pred = prediction.lower().strip()
distance = levenshtein(pred, truth)
cer = distance / max(len(truth), 1)
print("Truth:", truth)
print("Prediction:", pred)
print("Edit distance:", distance)
print("CER:", round(cer, 4))
print("MODEL CHECK COMPLETE")
